In [ ]:
import os
from tablevault import tablevault

vault = tablevault.Vault(user_id="jinjin",
                            process_name="quora_cross_encoder_symmetric_average_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [1]:
import torch
import numpy as np
from datasets import Dataset
from sentence_transformers import CrossEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report



In [ ]:
def softmax_np(x, axis=-1):
    x = np.asarray(x)
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def sigmoid_np(x):
    x = np.asarray(x)
    x = np.clip(x, -50, 50)
    return 1.0 / (1.0 + np.exp(-x))

def to_positive_prob(outputs):
    arr = np.asarray(outputs)
    if arr.ndim == 1:
        return sigmoid_np(arr)
    if arr.ndim == 2 and arr.shape[1] == 1:
        return sigmoid_np(arr[:, 0])
    if arr.ndim == 2 and arr.shape[1] >= 2:
        return softmax_np(arr, axis=1)[:, 1]
    raise ValueError(f"Unexpected prediction shape: {arr.shape}")


In [2]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

model_name = "cross-encoder/quora-distilroberta-base"
model = CrossEncoder(model_name, device=str(device))
print(model_name)
print("num_labels:", model.config.num_labels)


device: mps


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/quora-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


cross-encoder/quora-distilroberta-base
num_labels: 1


In [3]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(ds))
print("positive_rate:", y_true.mean())

original_pairs = list(zip(sent1, sent2))
reversed_pairs = list(zip(sent2, sent1))
print("first_original_pair:", original_pairs[0])
print("first_reversed_pair:", reversed_pairs[0])


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
first_original_pair: ("He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", '" The foodservice pie business does not fit our long-term growth strategy .')
first_reversed_pair: ('" The foodservice pie business does not fit our long-term growth strategy .', "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .")


In [4]:
batch_size = 64

orig_raw = model.predict(
    original_pairs,
    batch_size=batch_size,
    show_progress_bar=True,
    activation_fct=torch.nn.Identity(),
    convert_to_numpy=True,
)

rev_raw = model.predict(
    reversed_pairs,
    batch_size=batch_size,
    show_progress_bar=True,
    activation_fct=torch.nn.Identity(),
    convert_to_numpy=True,
)

orig_score = to_positive_prob(orig_raw)
rev_score = to_positive_prob(rev_raw)
avg_score = (orig_score + rev_score) / 2.0
y_pred = (avg_score >= 0.5).astype(int)

print("orig_raw_shape:", np.asarray(orig_raw).shape)
print("rev_raw_shape:", np.asarray(rev_raw).shape)
print("done")


The CrossEncoder.predict `activation_fct` argument was renamed and is now deprecated, please use `activation_fn` instead.


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

The CrossEncoder.predict `activation_fct` argument was renamed and is now deprecated, please use `activation_fn` instead.


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

orig_raw_shape: (408,)
rev_raw_shape: (408,)
done


In [ ]:

vault.create_record_list("distilbert_symmetric_average_prediction", column_names=["y_pred", "orig_score" , "rev_score"])

for i in range(len(y_pred)):
    vault.append_record("distilbert_symmetric_average_prediction", 
                        {
                            "y_pred": y_pred[i],
                            "orig_score": float(orig_score[i]),
                            "rev_score": float(rev_score[i]),
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "INSERT TEXT HERE ABOUT distilbert_symmetric_average_prediction"
embedding = get_embeddings(description)
vault.create_description("distilbert_symmetric_average_prediction", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert_symmetric_average_prediction", cat, embedding, prop)

In [5]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=['not_paraphrase', 'paraphrase'])

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


{'accuracy': 0.6985294117647058, 'f1': 0.7807486631016043}
                precision    recall  f1-score   support

not_paraphrase       0.52      0.51      0.52       129
    paraphrase       0.78      0.78      0.78       279

      accuracy                           0.70       408
     macro avg       0.65      0.65      0.65       408
  weighted avg       0.70      0.70      0.70       408



In [6]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))
    print("forward_score:", float(orig_score[i]))
    print("reverse_score:", float(rev_score[i]))
    print("avg_score:", float(avg_score[i]))


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 1
forward_score: 0.9779019355773926
reverse_score: 0.9833046197891235
avg_score: 0.9806032776832581
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 0
forward_score: 0.006291279569268227
reverse_score: 0.005755130667239428
avg_score: 0.0060232048854231834
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 0
forward_score: 0.04939701

In [7]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))
    print("forward_score:", float(orig_score[i]))
    print("reverse_score:", float(rev_score[i]))
    print("avg_score:", float(avg_score[i]))

vault.create_record_list("quora_cross_encoder_symmetric_average_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("quora_cross_encoder_symmetric_average_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "distilbert_symmetric_average_prediction": [0, len(ds)]
                    })

summary

description = "INSERT TEXT HERE ABOUT quora_cross_encoder_symmetric_average_mrpc_summary"
embedding = get_embeddings(description)
vault.create_description("quora_cross_encoder_symmetric_average_mrpc_summary", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("quora_cross_encoder_symmetric_average_mrpc_summary", cat, embedding, prop)



num_errors: 123
idx: 5
sentence1: Wal-Mart said it would check all of its million-plus domestic workers to ensure they were legally employed .
sentence2: It has also said it would review all of its domestic employees more than 1 million to ensure they have legal status .
true: 1 pred: 0
forward_score: 0.015424183569848537
reverse_score: 0.01830434799194336
avg_score: 0.016864266246557236
idx: 6
sentence1: While dioxin levels in the environment were up last year , they have dropped by 75 percent since the 1970s , said Caswell .
sentence2: The Institute said dioxin levels in the environment have fallen by as much as 76 percent since the 1970s .
true: 0 pred: 1
forward_score: 0.8578875660896301
reverse_score: 0.8719676733016968
avg_score: 0.8649276494979858
idx: 7
sentence1: This integrates with Rational PurifyPlus and allows developers to work in supported versions of Java , Visual C # and Visual Basic .NET.
sentence2: IBM said the Rational products were also integrated with Rational Pur

{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'cross-encoder/quora-distilroberta-base',
 'device': 'mps',
 'num_examples': 408,
 'aggregation': 'forward_reverse_average',
 'accuracy': 0.6985294117647058,
 'f1': 0.7807486631016043}

In [ ]:
description = "INSERT TEXT HERE ABOUT quora_cross_encoder_symmetric_average_mrpc" # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("quora_cross_encoder_symmetric_average_mrpc", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("quora_cross_encoder_symmetric_average_mrpc", cat, embedding, prop)